# New Guinea Orchid Identifier v2 — Colab Test
**EfficientNet-B4 genus classifier + FAISS species retrieval**

- No training needed — loads best_model.pth from Google Drive
- Runs Gradio UI with public share link
- Top-5 genera · Top-5 species

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q timm faiss-gpu-cu12 gradio

In [ ]:
# ── 2. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 3. Paths ──────────────────────────────────────────────────────────────────
from pathlib import Path

DRIVE_PROJECT = '/content/drive/MyDrive/orchid_project'
MODEL_DIR     = Path(f'{DRIVE_PROJECT}/models/efficientnet_b4')
FAISS_PATH    = Path(f'{DRIVE_PROJECT}/models/ong_species_index.faiss')
META_PATH     = Path(f'{DRIVE_PROJECT}/models/ong_metadata.json')
PHOTOS_DIR    = Path('/content/photos')   # extracted from photos.zip to local SSD

# Verify model files exist on Drive
for f in [MODEL_DIR/'best_model.pth', MODEL_DIR/'vocab.json', FAISS_PATH, META_PATH]:
    status = 'OK' if f.exists() else 'MISSING'
    print(f'[{status}] {f}')

In [ ]:
# ── 3b. Unzip photos (skip if already extracted) ──────────────────────────────
import os, shutil

PHOTOS_ZIP = f'{DRIVE_PROJECT}/photos.zip'
existing   = [d for d in os.scandir('/content/photos') if d.is_dir()] if os.path.exists('/content/photos') else []

if len(existing) >= 127:
    print(f'Photos already extracted: {len(existing)} genus folders found — skipping unzip.')
else:
    print('Unzipping photos.zip to /content/photos  (~9 GB, takes 3-5 min)...')
    if os.path.exists('/content/photos'):
        shutil.rmtree('/content/photos')
    os.makedirs('/content/photos', exist_ok=True)
    !unzip -q "{PHOTOS_ZIP}" -d /content/photos

    # Fix nested folder if unzip created photos/photos/
    nested = '/content/photos/photos'
    if os.path.isdir(nested):
        for item in os.listdir(nested):
            shutil.move(os.path.join(nested, item), '/content/photos')
        os.rmdir(nested)
        print('Fixed nested folder.')

    items = [d for d in os.scandir('/content/photos') if d.is_dir()]
    total = sum(len(os.listdir(d.path)) for d in items)
    print(f'Done! {len(items)} genera, {total:,} photos')

In [ ]:
# ── 4. Load model ─────────────────────────────────────────────────────────────
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import faiss
from torchvision import transforms
from PIL import Image
from collections import defaultdict

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

with open(MODEL_DIR / 'vocab.json') as f:
    genera = json.load(f)
n_cls = len(genera)
print(f'Classes: {n_cls} genera')

model = timm.create_model('efficientnet_b4', pretrained=False, num_classes=n_cls)
model.load_state_dict(
    torch.load(MODEL_DIR / 'best_model.pth', map_location=DEVICE, weights_only=False)
)
model = model.to(DEVICE).eval()
print('Model loaded OK')

In [ ]:
# ── 5. Load FAISS index + metadata ───────────────────────────────────────────
faiss_index = faiss.read_index(str(FAISS_PATH))

with open(META_PATH, encoding='utf-8') as f:
    metadata = json.load(f)

genus_to_idx = defaultdict(list)
for i, rec in enumerate(metadata):
    genus_to_idx[rec['genus']].append(i)

print(f'FAISS vectors : {faiss_index.ntotal:,}')
print(f'Metadata rows : {len(metadata):,}')

In [ ]:
# ── 6. Transform + inference functions ───────────────────────────────────────
tfm = transforms.Compose([
    transforms.Resize(int(380 * 1.1)),
    transforms.CenterCrop(380),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

PLACEHOLDER = Image.new('RGB', (300, 300), (220, 230, 220))


def resolve_photo(meta_path: str) -> Image.Image:
    """Map metadata path (Colab/Windows) to Drive path and open image."""
    parts = meta_path.replace('\\', '/').split('/')
    try:
        pi = [p.lower() for p in parts].index('photos')
        rel = '/'.join(parts[pi + 1:])          # genus/filename.jpg
        full = PHOTOS_DIR / rel
        if full.exists():
            return Image.open(full).convert('RGB')
    except (ValueError, Exception):
        pass
    return PLACEHOLDER


def identify(img: Image.Image, top_n=5, photos_per_species=5):
    tensor = tfm(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits         = model(tensor)
        probs          = F.softmax(logits, dim=1)[0]
        top5_p, top5_i = probs.topk(5)
        top3_p, top3_i = probs.topk(3)

        orig_head        = model.classifier
        model.classifier = nn.Identity()
        emb              = model(tensor).cpu().numpy().astype('float32')
        model.classifier = orig_head

    faiss.normalize_L2(emb)

    species_pool = defaultdict(list)
    for g_prob, g_idx in zip(top3_p.tolist(), top3_i.tolist()):
        genus     = genera[g_idx]
        g_indices = genus_to_idx.get(genus, [])
        if not g_indices:
            continue
        g_vecs  = np.array([faiss_index.reconstruct(i) for i in g_indices], dtype='float32')
        sub_idx = faiss.IndexFlatIP(g_vecs.shape[1])
        sub_idx.add(g_vecs)
        k       = min(50, len(g_indices))
        D, I    = sub_idx.search(emb, k)
        for score, local_i in zip(D[0], I[0]):
            rec      = metadata[g_indices[local_i]]
            combined = g_prob * float(score)
            species_pool[rec['species']].append({
                'genus':      genus,
                'genus_conf': g_prob,
                'score':      combined,
                'path':       rec['path'],
            })

    for sp in species_pool:
        species_pool[sp].sort(key=lambda x: x['score'], reverse=True)
        species_pool[sp] = species_pool[sp][:photos_per_species]

    ranked = sorted(
        species_pool.items(),
        key=lambda kv: kv[1][0]['score'],
        reverse=True
    )[:top_n]

    results = []
    for sp, photos in ranked:
        results.append({
            'species':    sp,
            'genus':      photos[0]['genus'],
            'genus_conf': photos[0]['genus_conf'],
            'best_score': photos[0]['score'],
            'photos':     photos,
        })

    genus_labels = {genera[i.item()]: float(p.item()) for p, i in zip(top5_p, top5_i)}
    return results, genus_labels


def predict(img):
    if img is None:
        return {}, [], 'Upload a photo to begin.'

    img = img.convert('RGB')
    results, genus_labels = identify(img)

    gallery = []
    for rank, res in enumerate(results, 1):
        for j, photo in enumerate(res['photos'], 1):
            caption = (
                f"#{rank} {res['species']}  [{j}/{len(res['photos'])}]\n"
                f"Genus: {res['genus']} ({res['genus_conf']*100:.0f}%)  |  "
                f"Score: {photo['score']:.3f}"
            )
            gallery.append((resolve_photo(photo['path']), caption))

    best    = results[0] if results else None
    summary = (
        f"<div style='text-align:center; padding:8px'>"
        f"<p style='font-size:1.5em; font-weight:bold; margin:0'>{best['species']}</p>"
        f"<p style='margin:4px 0 0 0; color:#555'>Genus: {best['genus']} · "
        f"confidence {best['genus_conf']*100:.0f}% · score {best['best_score']:.3f}</p>"
        f"</div>"
        if best else 'No results.'
    )
    return genus_labels, gallery, summary


print('Functions ready.')

In [ ]:
# ── 7. Quick single-image test (optional) ─────────────────────────────────────
# Upload any orchid photo to /content/ first, then change the path below
# test_img = Image.open('/content/your_orchid.jpg').convert('RGB')
# results, genus_labels = identify(test_img)
# print('Top genus:', max(genus_labels, key=genus_labels.get))
# for r in results:
#     print(f"  {r['species']:40s}  genus_conf={r['genus_conf']:.2f}  score={r['best_score']:.3f}")
print('(Uncomment lines above to test a single image)')

In [ ]:
# ── 8. Launch Gradio ──────────────────────────────────────────────────────────
import gradio as gr

with gr.Blocks(title='New Guinea Orchid Identifier v2 — Colab Test') as demo:
    gr.Markdown(
        '# New Guinea Orchid Identifier v2 — Colab Test\n'
        '**EfficientNet-B4 · 127 genera · 16,840 live field photos**  '
        '*Top-1 acc: 71% · Top-5 acc: 87%*\n\n'
        'Stage 1 predicts genus · Stage 2 returns top-5 species × 5 similar photos'
    )
    with gr.Row():
        with gr.Column(scale=1):
            img_input   = gr.Image(type='pil', label='Upload Orchid Photo')
            run_btn     = gr.Button('Identify', variant='primary')
            summary_out = gr.Markdown(label='Best Match')
        with gr.Column(scale=2):
            genus_out = gr.Label(num_top_classes=5, label='Stage 1 — Top-5 Genera')

    gallery_out = gr.Gallery(
        label='Stage 2 — Top-5 Species × 5 Similar Photos',
        columns=5, rows=5, height=650, object_fit='cover'
    )

    run_btn.click(fn=predict, inputs=img_input,
                  outputs=[genus_out, gallery_out, summary_out])
    img_input.change(fn=predict, inputs=img_input,
                     outputs=[genus_out, gallery_out, summary_out])

demo.launch(share=True, debug=False)  # share=True → public URL valid 72 hours